# Lab 03 — Linear Models

**Solved notebook.** Every task is solved and explained: a *How we got there* cell before each function gives the
reasoning, and an *Answers* cell after each experiment answers the guide's questions with the numbers this notebook
produces. Try the tasks in `lab03_linear_models.ipynb` first, then compare.

| Part | Task | sklearn twin | Time |
|:--|:--|:--|:-:|
| A | Data visualization: the target, the features, the relations | `seaborn`, `matplotlib` | 10 min |
| B | Evaluation methodology: a stratified split, regression and classification metrics from scratch | `train_test_split`, `metrics` | 20 min |
| C | Preprocessing: scaling fitted on the training set only, the logarithm | `StandardScaler` | 10 min |
| D | Linear regression: MSE loss, `jax.grad`, normal equation | `LinearRegression` | 15 min |
| E | Polynomial features, ridge, lasso and a proximal step | `PolynomialFeatures`, `Ridge`, `Lasso` | 30 min |
| F | The perceptron: its update is a `jax.grad` | `Perceptron` | 15 min |
| G | Logistic regression on spam, thresholds and metrics | `LogisticRegression` | 20 min |
| H | Challenge: one pipeline, three models | `Pipeline`, `cross_val_score` | home |

In [ ]:
import itertools

import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from lab03_utils import (
    compare,
    kfold_indices,
    load_housing,
    load_spam,
    make_blobs,
    make_xor,
    plot_boundary,
    plot_confusion,
    plot_cv_scores,
    plot_degree_curve,
    plot_history,
    plot_path,
    plot_predictions,
    plot_thresholds,
    plot_validation_curve,
    plot_weights,
    train,
)
from sklearn import metrics as skm
from sklearn.linear_model import Lasso, LinearRegression, LogisticRegression, Perceptron, Ridge
from sklearn.preprocessing import PolynomialFeatures

plt.rcParams["figure.figsize"] = (9, 4)
rng = np.random.default_rng(0)

## The two datasets

* **House prices** (regression): California housing, 20640 districts, 8 features, target = median house value in
  100 000 USD. Four skewed features are already replaced by their logarithm in `load_housing()`.
* **Spam** (classification): Spambase, 4601 e-mails, 57 features (word and character frequencies, capital-run lengths),
  39% spam. `load_spam()` returns the raw frequencies; `y = 1` is spam.

In [ ]:
X_h, y_h, names_h = load_housing()
X_s, y_s, names_s = load_spam()
print("housing:", X_h.shape, "target mean", y_h.mean().round(3), "| spam:", X_s.shape, "spam fraction", y_s.mean().round(3))

## Part A — Data visualization

Look before you model: the **target**, the **features** and the **relations** between them. The plots are ready: run
them, read them, and answer the questions of the guide. Every plot should end in a **decision** for the evaluation
methodology (Part B) or for the preprocessing (Part C).

### A1. The target

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
sns.histplot(y_h, bins=40, ax=axes[0])
axes[0].set(xlabel="median house value (100 000 USD)", title=f"housing: {int((y_h >= 5.0).sum())} districts at the cap 5.0")
sns.barplot(x=["ham", "spam"], y=[1 - y_s.mean(), y_s.mean()], ax=axes[1])
axes[1].set(ylabel="fraction of e-mails", title=f"spam: {y_s.mean():.1%} spam")
plt.tight_layout()
plt.show()

### A2. The features

Housing: the four skewed features in their raw form (top) and after the logarithm applied by `load_housing()` (bottom).
Spam: how many entries are exactly 0, and the capital-run length of the e-mails of each class.

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(13, 5))
for k, j in enumerate((2, 3, 4, 5)):
    sns.histplot(np.exp(X_h[:, j]), bins=40, ax=axes[0, k])
    axes[0, k].set(title=f"{names_h[j]}, raw", xlabel="")
    sns.histplot(X_h[:, j], bins=40, ax=axes[1, k])
    axes[1, k].set(title=f"{names_h[j]}, log", xlabel="")
plt.tight_layout()
plt.show()

cap = names_s.index("capital_run_length_total")
fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
sns.histplot((X_s == 0).mean(axis=0), bins=20, ax=axes[0])
axes[0].set(xlabel="fraction of e-mails where the feature is exactly 0", ylabel="features", title="spam: most features are mostly 0")
sns.histplot(x=np.log10(X_s[:, cap]), hue=np.where(y_s == 1, "spam", "ham"), bins=30, stat="percent", common_norm=False, ax=axes[1])
axes[1].set(xlabel="log10 of capital letters (total)", title="a long right tail")
plt.tight_layout()
plt.show()

### A3. The relations

Housing: the correlation matrix and a map (colour = price). Spam: the ten features whose mean $\log(1+x)$ differs most between
the two classes.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))
sns.heatmap(np.corrcoef(np.c_[X_h, y_h].T), annot=True, fmt=".2f", cmap="vlag", center=0, cbar=False,
            xticklabels=[*names_h, "target"], yticklabels=[*names_h, "target"], ax=axes[0])
sns.scatterplot(x=X_h[:, names_h.index("Longitude")], y=X_h[:, names_h.index("Latitude")], hue=y_h, palette="viridis", s=4, linewidth=0, legend=False, ax=axes[1])
axes[1].set(xlabel="longitude", ylabel="latitude", title="housing: prices follow the coast")
gap_s = np.log1p(X_s[y_s == 1]).mean(axis=0) - np.log1p(X_s[y_s == 0]).mean(axis=0)
top = np.argsort(-np.abs(gap_s))[:10]
sns.barplot(x=gap_s[top], y=[names_s[j] for j in top], ax=axes[2])
axes[2].set(xlabel="mean log(1 + x): spam minus ham", title="spam: the ten most discriminating features")
plt.tight_layout()
plt.show()

### Answers (A1–A3)

1. **The target (A1).** The housing target has a **wall at 5.0**: 992 districts (4.8%) are capped, so the true prices above
   500 000 USD are unknown and no model can predict them: a floor for the test error and a flat stripe in every
   predicted-versus-actual plot. The spam target is 39.4% spam: imbalanced but not rare, which is why we stratify (B1) and
   why accuracy alone is not enough (B3).
2. **The features (A2).** Four housing features are extremely right-skewed in their raw form (skewness from 5 to 98;
   `AveOccup` has a maximum 441 times its median), so a few districts would dominate a linear fit: hence the logarithm,
   applied by `load_housing()` (`AveBedrms` stays skewed, 6.4, even after it). In the spam data **77.4%** of the entries are
   exactly 0 (19 of the 57 features are 0 in more than 90% of the e-mails) and the capital-run length spans orders of
   magnitude (skewness 8.7): $\log(1+x)$ keeps 0 at 0 and removes the tail (C2).
3. **The relations (A3).** `MedInc` is the strongest feature of the price ($r = 0.69$), the others have $|r| \le 0.26$.
   Latitude and longitude are almost collinear ($r = -0.92$) and act **jointly** on the map (the coast, the Bay Area, Los
   Angeles): a plane in them is a poor model, which polynomial features partly repair (Part E). In spam the capital-run
   lengths, `your`, `you`, `!` and `free` point to spam, while `hp` and `george` point to ham: they come from one HP Labs
   mailbox, an artefact that will not transfer to other mailboxes.

## Part B — Evaluation methodology

Before any model, fix **how it will be judged**: a split that respects the classes (B1) and metrics that match the
problem, written from scratch (B2 for regression, B3 for classification).

### B1. A stratified split

Implement `stratified_split(y, test_frac, rng)`: return `(train_idx, test_idx)` such that **each class** keeps the
same proportion in both sets (guide, task B1).

### How we got there (B1)

A plain random split can put too many (or too few) spam e-mails in the test set, especially when the set is small. To
stratify, we split **each class separately**: shuffle the indices of class $c$ (`rng.permutation`), and send the first
`round(n_c * test_frac)` of them to the test set. The training indices are everything else (`np.setdiff1d`). Both index
arrays are shuffled once more, so the classes are not stored in blocks.

In [ ]:
def stratified_split(y, test_frac, rng) -> tuple[np.ndarray, np.ndarray]:
    test = []
    for c in np.unique(y):
        idx = rng.permutation(np.flatnonzero(y == c))
        test.append(idx[: round(len(idx) * test_frac)])
    test = np.concatenate(test)
    train_idx = np.setdiff1d(np.arange(len(y)), test)
    return rng.permutation(train_idx), rng.permutation(test)

In [ ]:
# Housing: an ordinary random 80/20 split. Spam: stratified 80/20.
perm = rng.permutation(len(y_h))
te_h, tr_h = perm[: len(perm) // 5], perm[len(perm) // 5 :]
tr_s, te_s = stratified_split(y_s, 0.2, rng)
ytr_h, yte_h = y_h[tr_h], y_h[te_h]
ytr_s, yte_s = y_s[tr_s], y_s[te_s]
print("housing train/test:", len(tr_h), len(te_h))
print("spam train/test:", len(tr_s), len(te_s), "| spam fraction train", y_s[tr_s].mean().round(4), "test", y_s[te_s].mean().round(4))

# Why stratify? On 100 e-mails, how different are the spam fractions of the two parts?
def gap(stratify, n_rep=300):
    gaps = []
    for _ in range(n_rep):
        sub = y_s[rng.choice(len(y_s), 100, replace=False)]
        tr, te = stratified_split(sub, 0.2, rng) if stratify else (lambda p: (p[20:], p[:20]))(rng.permutation(100))
        gaps.append(abs(sub[tr].mean() - sub[te].mean()))
    return np.array(gaps)

gap_random, gap_strat = gap(False), gap(True)
print(f"mean |train - test| spam fraction: random {gap_random.mean():.3f}, stratified {gap_strat.mean():.3f}")
_, ax = plt.subplots(figsize=(7, 3))
sns.histplot({"random split": gap_random, "stratified split": gap_strat}, bins=20, ax=ax)
ax.set(xlabel="|spam fraction of train - spam fraction of test|, on 100 e-mails", title="why stratify")
plt.show()

### Answers (B1)

1. On 100 e-mails, a random split leaves an average gap of **0.092** between the spam fraction of the two parts, the
   stratified split only **0.015**. Stratifying removes the sampling noise of the class proportions: every part is a
   miniature of the whole (on the full spam data the fractions are 0.394 and 0.3941). The effect vanishes for large,
   balanced sets, and it matters most for **rare classes** and small test sets.

### B2. Regression metrics

Implement `regression_metrics(y, y_hat)` returning a dictionary with `mse`, `rmse`, `mae`, `smape` (in %) and `r2`:

$$\text{sMAPE} = \frac{100}{n}\sum_i \frac{2\,|y_i - \hat y_i|}{|y_i| + |\hat y_i|} \qquad
R^2 = 1 - \frac{\sum_i (y_i - \hat y_i)^2}{\sum_i (y_i - \bar y)^2}$$

### How we got there (B2)

With `e = y - y_hat`: MSE is `mean(e**2)`, RMSE its square root (same unit as $y$), MAE `mean(abs(e))`. sMAPE divides
the absolute error by the *mean* of the two magnitudes (that is the factor 2 in the numerator), so it stays finite when
one of them is small, unlike the plain percentage error. $R^2$ compares the model's squared error with that of the
best constant predictor $\bar y$: 1 is perfect, 0 is no better than the mean, and it can be **negative**.

In [ ]:
def regression_metrics(y, y_hat) -> dict[str, float]:
    e = y - y_hat
    return {
        "mse": float(np.mean(e**2)),
        "rmse": float(np.sqrt(np.mean(e**2))),
        "mae": float(np.mean(np.abs(e))),
        "smape": float(100 * np.mean(2 * np.abs(e) / (np.abs(y) + np.abs(y_hat)))),
        "r2": float(1 - np.sum(e**2) / np.sum((y - y.mean()) ** 2)),
    }

In [ ]:
# Check against scikit-learn (sMAPE has no sklearn function: verify it by hand on two points).
y_t, y_p = np.array([100.0, 200.0, 50.0, 80.0]), np.array([110.0, 180.0, 55.0, 60.0])
ours = regression_metrics(y_t, y_p)
compare("mse", ours["mse"], skm.mean_squared_error(y_t, y_p))
compare("mae", ours["mae"], skm.mean_absolute_error(y_t, y_p))
compare("r2", ours["r2"], skm.r2_score(y_t, y_p))
print("smape:", round(ours["smape"], 4), "by hand:", round(100 * np.mean([20 / 210, 40 / 380, 10 / 105, 40 / 140]), 4))
print("baseline (predict the training mean) on the housing test set:")
print({k: round(v, 3) for k, v in regression_metrics(yte_h, np.full(len(yte_h), ytr_h.mean())).items()})

### B3. Classification metrics

Implement `confusion_counts(y, y_hat)` returning `(tn, fp, fn, tp)` and `classification_metrics(y, y_hat)` returning
`accuracy`, `precision`, `recall`, `f1` and `mcc`. Return 0 when a denominator is 0.

$$\text{MCC} = \frac{TP\cdot TN - FP\cdot FN}{\sqrt{(TP+FP)(TP+FN)(TN+FP)(TN+FN)}}$$

### How we got there (B3)

Each cell of the confusion matrix is a count of a Boolean combination: $TP$ = actual 1 **and** predicted 1, and so on.
Precision, recall and $F_1$ are ratios of those counts, so we guard every division. MCC is the correlation between
the predicted and the actual labels: it uses all four cells, so it is high only when the model is good on **both**
classes. Cast the counts to `float` before the square root, to avoid integer overflow on large sets.

In [ ]:
def confusion_counts(y, y_hat) -> tuple[int, int, int, int]:
    tp = int(np.sum((y == 1) & (y_hat == 1)))
    tn = int(np.sum((y == 0) & (y_hat == 0)))
    fp = int(np.sum((y == 0) & (y_hat == 1)))
    fn = int(np.sum((y == 1) & (y_hat == 0)))
    return tn, fp, fn, tp


def classification_metrics(y, y_hat) -> dict[str, float]:
    tn, fp, fn, tp = confusion_counts(y, y_hat)
    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    denom = np.sqrt(float(tp + fp) * (tp + fn) * (tn + fp) * (tn + fn))
    mcc = (tp * tn - fp * fn) / denom if denom else 0.0
    return {"accuracy": (tp + tn) / len(y), "precision": precision, "recall": recall, "f1": f1, "mcc": float(mcc)}

In [ ]:
y_t = np.array([1, 1, 1, 1, 0, 0, 0, 0, 0, 0])
y_p = np.array([1, 1, 1, 0, 1, 0, 0, 0, 0, 0])
ours = classification_metrics(y_t, y_p)
compare("confusion matrix", confusion_counts(y_t, y_p), skm.confusion_matrix(y_t, y_p).ravel())
compare("accuracy", ours["accuracy"], skm.accuracy_score(y_t, y_p))
compare("precision", ours["precision"], skm.precision_score(y_t, y_p))
compare("recall", ours["recall"], skm.recall_score(y_t, y_p))
compare("f1", ours["f1"], skm.f1_score(y_t, y_p))
compare("mcc", ours["mcc"], skm.matthews_corrcoef(y_t, y_p))

# A useless model: call every e-mail spam.
all_spam = classification_metrics(yte_s, np.ones_like(yte_s))
print({k: round(v, 3) for k, v in all_spam.items()})

fig, axes = plt.subplots(1, 2, figsize=(7, 3))
plot_confusion(*confusion_counts(y_t, y_p), ax=axes[0], title="the toy example")
plot_confusion(*confusion_counts(yte_s, np.ones_like(yte_s)), ax=axes[1], title='"every e-mail is spam"')
plt.tight_layout()
plt.show()

### Answers (B2–B3)

* **B2.** The functions agree with scikit-learn to the last digit, and sMAPE (14.54%) agrees with the hand computation.
  The baseline that always predicts the training mean has **$R^2 = 0$** by definition, RMSE 1.17 (117 000 USD), MAE
  0.925 and sMAPE 45.7%. Every model must beat those. sMAPE is scale-free, but it is unstable when the true value and
  the prediction are both near 0 (the denominator vanishes); house prices are far from 0, so it is safe here.
* **B3.** Our confusion matrix and all five metrics equal scikit-learn's. The "everything is spam" filter gets accuracy
  **0.394** (the spam fraction), precision 0.394, recall **1.0** and $F_1 = 0.565$: recall and $F_1$ make a useless
  model look acceptable, because they ignore the true negatives. **MCC = 0** says the truth: no better than chance. MCC
  uses all four cells of the matrix, so it cannot be inflated by predicting one class.

## Part C — Preprocessing

Whatever is learned from the data (a mean, a deviation, a vocabulary) is fitted on the **training part only** and applied
unchanged to the test part (C1). Skewed features need a transformation, chosen from what Part A showed (C2).

### C1. Scaling fitted on the training set only

Implement `fit_standardizer(X)` (per-column mean and standard deviation; a constant column gets standard deviation 1)
and `standardize(X, mean, std)`. The mean and the deviation are **learned from the training set** and applied to both
sets. Fitting them on all the data leaks information from the test set (Class 01, data leakage).

### How we got there (C1)

`X.mean(axis=0)` and `X.std(axis=0)` are computed on the training rows only. A constant column would divide by zero,
so its deviation is replaced by 1 (`np.where`). `standardize` is then pure arithmetic and works on any set. The spam
frequencies are extremely skewed (most values are 0, a few are large), so we apply `np.log1p` first and standardize the
result.

In [ ]:
def fit_standardizer(X) -> tuple[np.ndarray, np.ndarray]:
    std = X.std(axis=0)
    return X.mean(axis=0), np.where(std == 0, 1.0, std)


def standardize(X, mean, std) -> np.ndarray:
    return (X - mean) / std

In [ ]:
mean_h, std_h = fit_standardizer(X_h[tr_h])
Xtr_h, Xte_h = standardize(X_h[tr_h], mean_h, std_h), standardize(X_h[te_h], mean_h, std_h)

L_s = np.log1p(X_s)
mean_s, std_s = fit_standardizer(L_s[tr_s])
Xtr_s, Xte_s = standardize(L_s[tr_s], mean_s, std_s), standardize(L_s[te_s], mean_s, std_s)
print("train means ~0:", Xtr_h.mean(axis=0).round(2)[:4], "| test means differ a little:", Xte_h.mean(axis=0).round(2)[:4])

fig, axes = plt.subplots(1, 2, figsize=(12, 3.8), sharey=True)
sns.boxplot(data=X_h[tr_h], orient="h", fliersize=1, ax=axes[0])
axes[0].set(yticks=range(8), yticklabels=names_h, title="raw features: very different scales")
sns.boxplot(data=Xtr_h, orient="h", fliersize=1, ax=axes[1])
axes[1].set(title="standardized with the training statistics")
plt.tight_layout()
plt.show()

### Answers (C1)

The training means are 0 by construction; the test means are close but not exactly 0 (0.01, 0.03, ...): the test
  set was scaled with the training statistics, as a deployed model would be. Fitting the mean and deviation on all
  the data would let the test set influence the transformation (leakage), which is small here and large in other problems.

### C2. The logarithm of a skewed feature

The spam frequencies are mostly 0 with a long right tail. $\log(1+x)$ keeps 0 at 0 and compresses the tail; the
standardizer of C1 then puts the result on a common scale. The cell shows the effect on `capital_run_length_total`.

In [ ]:
def skewness(a):
    return float(np.mean((a - a.mean()) ** 3) / a.std() ** 3)


cap = names_s.index("capital_run_length_total")
fig, axes = plt.subplots(1, 3, figsize=(13, 3.4))
for ax, (title, values) in zip(axes, [("raw", X_s[tr_s, cap]), ("log(1 + x)", L_s[tr_s, cap]), ("log, then standardized", Xtr_s[:, cap])]):
    sns.histplot(values, bins=40, ax=ax)
    ax.set(title=f"{title}: skewness {skewness(values):.2f}", xlabel="capital_run_length_total")
plt.tight_layout()
plt.show()

### Answers (C2)

The raw feature has a skewness of 8.7 (a maximum 167 times the median); after $\log(1+x)$ it is 0.01: a roughly symmetric
bell. The standardizer only changes the scale (mean 0, deviation 1), never the shape, so the logarithm must come first.
Because the logarithm has no parameters learned from the data, applying it to all the rows leaks nothing; the mean and the
deviation after it do, and they come from the training part only.

## Part D — Linear regression

$\hat y = w^\top x + b$, trained by minimizing the mean squared error. Parameters are a dictionary
`{"w": (d,), "b": ()}`, the format `train` expects.

### D1. Model, loss and normal equation

Implement `init_params(d)` (zeros), `predict_linear(params, X)`, `mse_loss(params, X, y)` (written with `jax.numpy`;
**no gradient formulas**) and `normal_equation(X, y)`: append a column of ones to $X$ and solve
$(A^\top A)\,\theta = A^\top y$ with `np.linalg.solve`. It returns `(w, b)`.

### How we got there (D1)

The model is one matrix product plus the bias. The loss is the mean of the squared residuals: because it is written
with `jax.numpy`, `jax.value_and_grad` inside `train` differentiates it, so the derivation of the previous class
($\frac2N X^\top r$) is never typed. The normal equation is the place where that gradient is set to zero:
$A^\top(A\theta - y) = 0$, a linear system that `np.linalg.solve` handles without inverting the matrix.

In [ ]:
def init_params(d) -> dict[str, jax.Array]:
    return {"w": jnp.zeros(d), "b": jnp.zeros(())}


def predict_linear(params, X) -> jax.Array:
    return X @ params["w"] + params["b"]


def mse_loss(params, X, y) -> jax.Array:
    return jnp.mean((predict_linear(params, X) - y) ** 2)


def normal_equation(X, y) -> tuple[np.ndarray, float]:
    A = np.c_[X, np.ones(len(X))]
    theta = np.linalg.solve(A.T @ A, A.T @ y)
    return theta[:-1], theta[-1]

In [ ]:
params, hist = train(mse_loss, init_params(8), Xtr_h, ytr_h, lr=0.05, steps=1000)
sk = LinearRegression().fit(Xtr_h, ytr_h)
w_ne, b_ne = normal_equation(Xtr_h, ytr_h)

compare("jax.grad vs sklearn (weights)", params["w"], sk.coef_)
compare("jax.grad vs sklearn (bias)", params["b"], sk.intercept_)
compare("normal equation vs sklearn", np.r_[w_ne, b_ne], np.r_[sk.coef_, sk.intercept_])
for name, pred in [
    ("mean baseline", np.full(len(yte_h), ytr_h.mean())),
    ("ours (jax.grad)", np.asarray(predict_linear(params, Xte_h))),
    ("sklearn", sk.predict(Xte_h)),
]:
    print(f"{name:<16}", {k: round(v, 3) for k, v in regression_metrics(yte_h, pred).items()})

In [ ]:
plot_predictions(yte_h, np.asarray(predict_linear(params, Xte_h)), "linear regression trained with jax.grad (test set)")
plot_weights(names_h, {"jax.grad": params["w"], "scikit-learn": sk.coef_})

In [ ]:
# The same model on the raw (unstandardized) features, with the same optimizer and settings.
_, hist_raw = train(mse_loss, init_params(8), X_h[tr_h], ytr_h, lr=0.05, steps=1000)
plot_history({"standardized": hist, "raw features": hist_raw})
print("final training loss: standardized", hist[-1].round(4), "| raw", hist_raw[-1].round(4))

### Answers (D1)

1. **The three solutions coincide:** the weights of `jax.grad` and of scikit-learn differ by $10^{-15}$, and the normal
   equation by $2\cdot10^{-14}$. The test metrics are identical: **MSE 0.418, RMSE 0.646, MAE 0.478, sMAPE 26.3%,
   $R^2 = 0.695$**, against $R^2 = 0$ and RMSE 1.17 for the mean baseline. About 30% of the variance is left
   unexplained: with 8 features and a linear model, that is the model's limit (Part E tries more flexibility).
2. **Standardization changes the speed, not the answer.** After the same 1000 Adam steps, the standardized model reaches a
   training loss of 0.4554 (its optimum), the raw one 0.532. The raw features have very different scales (latitude near 35,
   longitude near $-120$, median income near 4), so the loss is a long, thin valley (Class 02: conditioning) and the
   optimizer needs many more steps. The closed-form solution does not care.
3. **The plots.** In predicted-versus-actual, the districts capped at 5.0 form a **vertical wall**: whatever the model
   predicts for them (between 2.5 and 7) is compared with 5.0, so the cap is a floor for the error. The residuals are centred
   on 0 with a long right tail: the expensive districts are under-predicted. In the bar chart the weights of `jax.grad` and of
   scikit-learn are indistinguishable; latitude and longitude carry the largest negative weights (the coast is expensive)
   and `MedInc` the largest positive one.

## Part E — Polynomial features and regularization

### E1. Polynomial feature expansion

Implement `polynomial_features(X, degree)`: all monomials of the columns with degrees $1,\dots,\text{degree}$
(no constant column), ordered by degree and, inside a degree, by `itertools.combinations_with_replacement`. Check it
against `PolynomialFeatures(degree, include_bias=False)`.

### How we got there (E1)

A monomial of degree $k$ is a product of $k$ columns, possibly repeated: $x_1^2 x_3$ is the multiset $(1, 1, 3)$. The
multisets of size $k$ are exactly `itertools.combinations_with_replacement(range(d), k)`, and each becomes one new column,
`np.prod(X[:, combo], axis=1)`. With $d = 8$ the number of columns is $\binom{d+k}{k} - 1$: 8, 44, 164, 494 for degrees
1 to 4. The model stays **linear in the parameters**, so everything from Part D still applies.

In [ ]:
def polynomial_features(X, degree) -> np.ndarray:
    columns = []
    for k in range(1, degree + 1):
        for combo in itertools.combinations_with_replacement(range(X.shape[1]), k):
            columns.append(np.prod(X[:, list(combo)], axis=1))
    return np.stack(columns, axis=1)

In [ ]:
for d in (2, 3):
    compare(f"degree {d} vs PolynomialFeatures", polynomial_features(X_h[:50], d), PolynomialFeatures(d, include_bias=False).fit_transform(X_h[:50]))
print("number of features for degrees 1-4:", [polynomial_features(X_h[:5], d).shape[1] for d in range(1, 5)])

Now the problem: with only **300 training examples**, fit the linear model on the expanded features for degrees 1 to 4
(scikit-learn's `LinearRegression` solves it exactly; the expanded columns are standardized on the training part).

In [ ]:
sub = rng.choice(tr_h, 300, replace=False)
train_mse, test_mse = [], []
for d in range(1, 5):
    P, Pt = polynomial_features(X_h[sub], d), polynomial_features(X_h[te_h], d)
    m_, s_ = fit_standardizer(P)
    model = LinearRegression().fit(standardize(P, m_, s_), y_h[sub])
    train_mse.append(regression_metrics(y_h[sub], model.predict(standardize(P, m_, s_)))["mse"])
    test_mse.append(regression_metrics(y_h[te_h], model.predict(standardize(Pt, m_, s_)))["mse"])
    print(f"degree {d}: {P.shape[1]:4d} features | train MSE {train_mse[-1]:.4f} | test MSE {test_mse[-1]:.4g}")
plot_degree_curve(range(1, 5), train_mse, test_mse)

### Answers (E1)

`polynomial_features` equals scikit-learn's (difference 0 for degree 2 and $10^{-10}$ for degree 3, rounding) and the
number of columns is 8, 44, 164, 494. With **300 training examples**:

| degree | features | train MSE | test MSE |
|:-:|--:|--:|--:|
| 1 | 8 | 0.4544 | 0.4508 |
| 2 | 44 | 0.3165 | 0.5328 |
| 3 | 164 | 0.1537 | 14.46 |
| 4 | 494 | 0.0010 | 27 380 |

The training error falls at every degree (more flexibility can only fit the training set better) and reaches almost 0
at degree 4, where there are more features (494) than examples (300): the model **interpolates**, noise included. The
test error is best for the plain line and then explodes: this is overfitting (Class 01, bias and variance), made
worse because a degree-4 monomial of an outlying district is enormous. The test error, not the training error, chooses
the degree.

### E2. Ridge (L2) and lasso (L1) losses

Implement `ridge_loss(params, X, y, lam)` $= \text{MSE} + \lambda\lVert w\rVert_2^2$ and
`lasso_loss(params, X, y, lam)` $= \text{MSE} + \lambda\lVert w\rVert_1$. The bias is **not** penalized. Pass
`lam` to `train` as data, so JAX compiles the loss once for all values of $\lambda$. Then choose $\lambda$ by 5-fold
cross-validation on the 300 examples (degree 2, 44 features) and compare with scikit-learn.

*Conventions:* scikit-learn's `Ridge(alpha)` minimizes $\lVert y - Xw\rVert^2 + \alpha\lVert w\rVert^2$ (a *sum*), so
$\alpha = \lambda N$. Its `Lasso(alpha)` minimizes $\frac1{2N}\lVert y - Xw\rVert^2 + \alpha\lVert w\rVert_1$, so
$\alpha = \lambda/2$.

### How we got there (E2)

Both losses are the `mse_loss` plus a penalty on `params["w"]` only (`jnp.sum(w**2)` and `jnp.sum(jnp.abs(w))`).
`jax.grad` differentiates `jnp.abs` too: away from 0 the gradient is $\text{sign}(w)$, and at exactly 0 JAX returns 0,
a valid *subgradient*. The conventions follow from writing the two objectives on the same footing: ridge
$\frac1N\lVert r\rVert^2 + \lambda\lVert w\rVert^2 = \frac1N(\lVert r\rVert^2 + \lambda N\lVert w\rVert^2)$, and the lasso
$\frac1N\lVert r\rVert^2 + \lambda\lVert w\rVert_1 = 2\big(\frac1{2N}\lVert r\rVert^2 + \frac\lambda2\lVert w\rVert_1\big)$.
Inside the cross-validation, the standardizer is fitted on each **fold's training part**, never on the validation part.

In [ ]:
def ridge_loss(params, X, y, lam) -> jax.Array:
    return mse_loss(params, X, y) + lam * jnp.sum(params["w"] ** 2)


def lasso_loss(params, X, y, lam) -> jax.Array:
    return mse_loss(params, X, y) + lam * jnp.sum(jnp.abs(params["w"]))

In [ ]:
P2, P2_test = polynomial_features(X_h[sub], 2), polynomial_features(X_h[te_h], 2)
lams = np.logspace(-4, 0.5, 10)
folds = kfold_indices(len(sub), 5, rng)

train_curve, val_curve = [], []
for lam in lams:
    tr_e, va_e = [], []
    for tr_i, va_i in folds:
        m_, s_ = fit_standardizer(P2[tr_i])
        p, _ = train(ridge_loss, init_params(P2.shape[1]), standardize(P2[tr_i], m_, s_), y_h[sub][tr_i], lam, lr=0.05, steps=1500)
        tr_e.append(regression_metrics(y_h[sub][tr_i], np.asarray(predict_linear(p, standardize(P2[tr_i], m_, s_))))["mse"])
        va_e.append(regression_metrics(y_h[sub][va_i], np.asarray(predict_linear(p, standardize(P2[va_i], m_, s_))))["mse"])
    train_curve.append(np.mean(tr_e))
    val_curve.append(np.mean(va_e))
best = lams[int(np.argmin(val_curve))]
plot_validation_curve(lams, train_curve, val_curve)
print(f"best lambda by 5-fold CV: {best:.4g}")

In [ ]:
# Refit on the 300 examples with the best lambda and compare with scikit-learn's Ridge (alpha = lambda * N).
m2, s2 = fit_standardizer(P2)
A2, A2_test = standardize(P2, m2, s2), standardize(P2_test, m2, s2)
ridge_p, _ = train(ridge_loss, init_params(44), A2, y_h[sub], best, lr=0.05, steps=3000)
ridge_sk = Ridge(alpha=best * len(sub)).fit(A2, y_h[sub])
compare("ridge weights", ridge_p["w"], ridge_sk.coef_)
lin_sk = LinearRegression().fit(A2, y_h[sub])
print("test MSE on 300 examples, degree 2: plain", round(regression_metrics(y_h[te_h], lin_sk.predict(A2_test))["mse"], 4),
      "| ridge", round(regression_metrics(y_h[te_h], np.asarray(predict_linear(ridge_p, A2_test)))["mse"], 4),
      "| best CV MSE", round(min(val_curve), 4))

# Regularization paths: coefficients versus lambda.
path = np.array([np.asarray(train(ridge_loss, init_params(44), A2, y_h[sub], lam, lr=0.05, steps=1500)[0]["w"]) for lam in lams])
plot_path(lams, path, "ridge (L2): coefficients shrink smoothly towards 0")

In [ ]:
# The same 300 examples with degree-3 features (164 columns): the plain fit exploded in E1, ridge tames it.
P3, P3_test = polynomial_features(X_h[sub], 3), polynomial_features(X_h[te_h], 3)
m3, s3 = fit_standardizer(P3)
B3, B3_test = standardize(P3, m3, s3), standardize(P3_test, m3, s3)
plain3 = LinearRegression().fit(B3, y_h[sub])
ridge3, _ = train(ridge_loss, init_params(164), B3, y_h[sub], 1.0, lr=0.05, steps=3000)
print("degree 3, 300 examples, test MSE: plain", round(regression_metrics(y_h[te_h], plain3.predict(B3_test))["mse"], 3),
      "| ridge (lambda = 1)", round(regression_metrics(y_h[te_h], np.asarray(predict_linear(ridge3, B3_test)))["mse"], 3))

In [ ]:
# Lasso by jax.grad (a subgradient): compare with scikit-learn's coordinate descent.
lam = 0.03
lasso_p, _ = train(lasso_loss, init_params(44), A2, y_h[sub], lam, lr=0.02, steps=6000)
lasso_sk = Lasso(alpha=lam / 2, max_iter=1_000_000, tol=1e-12).fit(A2, y_h[sub])


def objective(w, b):
    return float(np.mean((A2 @ w + b - y_h[sub]) ** 2) + lam * np.abs(w).sum())


compare("lasso weights", lasso_p["w"], lasso_sk.coef_)
compare("lasso objective value", objective(np.asarray(lasso_p["w"]), float(lasso_p["b"])), objective(lasso_sk.coef_, lasso_sk.intercept_))
compare("lasso test predictions", predict_linear(lasso_p, A2_test), lasso_sk.predict(A2_test))
w = np.asarray(lasso_p["w"])
print(f"exactly zero: ours {int(np.sum(w == 0))} | sklearn {int(np.sum(lasso_sk.coef_ == 0))} of 44")
print(f"below 1e-3:   ours {int(np.sum(np.abs(w) < 1e-3))} | sklearn {int(np.sum(np.abs(lasso_sk.coef_) < 1e-3))}")

### Answers (E2)

* **Ridge.** Our ridge equals scikit-learn's `Ridge(alpha = lambda * N)` to $3\cdot10^{-16}$. The 5-fold validation
  curve has its minimum at $\lambda = 1$ (validation MSE 0.646): too small a $\lambda$ leaves the overfitting, too large
  a one shrinks everything to the mean. On the 44 degree-2 features the test MSE goes from 0.533 (plain) to **0.510**.
  The gain is bigger where the plain fit failed badly: with the 164 degree-3 features the plain test MSE is **14.47**,
  the ridge one (also $\lambda = 1$) **0.554**. The penalty shrinks all coefficients smoothly (the path plot) and none
  reaches 0.
* **Lasso by `jax.grad`.** With $\lambda = 0.03$ the objective value and the predictions agree with scikit-learn's
  coordinate descent ($6\cdot10^{-6}$ and $8\cdot10^{-3}$), but the weights differ by up to 0.145 and, above all,
  **no weight is exactly 0** (31 are below $10^{-3}$, they keep oscillating around 0), while scikit-learn returns **32 exact
  zeros** out of 44. The subgradient jumps over the kink at 0. The weights differ because the polynomial features are
  nearly collinear (condition number $3\cdot10^{7}$): very different weight vectors give almost the same fit.

### E3. A proximal step gives exact zeros

Implement `soft_threshold(w, t)` $= \text{sign}(w)\max(|w| - t, 0)$ and `proximal_lasso(X, y, lam, lr, steps)`. Each
iteration takes a gradient step on the **smooth part** (`jax.grad(mse_loss)`), then soft-thresholds the weights by
$\text{lr}\cdot\lambda$ (the bias is not thresholded). This is ISTA, *proximal* gradient descent.

### How we got there (E3)

The $L_1$ penalty is not differentiable at 0, and a plain (sub)gradient step keeps jumping over 0. The proximal step
treats the two parts separately: descend on the smooth MSE with the gradient from `jax.grad`, then apply the
**proximal operator** of $\lambda\lVert\cdot\rVert_1$, which is exactly the soft threshold: any weight whose magnitude
is below $\text{lr}\cdot\lambda$ is set to 0, the others move towards 0 by that amount. Weights can therefore *stay*
at exactly zero.

In [ ]:
def soft_threshold(w, t) -> jax.Array:
    return jnp.sign(w) * jnp.maximum(jnp.abs(w) - t, 0.0)


def proximal_lasso(X, y, lam, lr, steps) -> dict[str, jax.Array]:
    grad = jax.jit(jax.grad(mse_loss))
    params = init_params(X.shape[1])
    for _ in range(steps):
        g = grad(params, X, y)
        params = {"w": soft_threshold(params["w"] - lr * g["w"], lr * lam), "b": params["b"] - lr * g["b"]}
    return params

In [ ]:
# The MSE gradient is L-Lipschitz with L = 2 * lambda_max(A^T A / N) (its Hessian): the step 1/L is the safe choice.
lipschitz = 2 * np.linalg.eigvalsh(A2.T @ A2 / len(A2)).max()
prox_p = proximal_lasso(A2, y_h[sub], lam, 1 / lipschitz, 100_000)
print(f"L = {lipschitz:.1f}, step 1/L = {1 / lipschitz:.4f}")
compare("proximal lasso weights", prox_p["w"], lasso_sk.coef_)
compare("proximal lasso objective value", objective(np.asarray(prox_p["w"]), float(prox_p["b"])), objective(lasso_sk.coef_, lasso_sk.intercept_))
compare("proximal lasso test predictions", predict_linear(prox_p, A2_test), lasso_sk.predict(A2_test))
print(f"exactly zero: proximal {int(np.sum(np.asarray(prox_p['w']) == 0))} | sklearn {int(np.sum(lasso_sk.coef_ == 0))} of 44")
kept = [i for i in range(44) if np.asarray(prox_p["w"])[i] != 0]
print("test MSE: lasso", round(regression_metrics(y_h[te_h], np.asarray(predict_linear(prox_p, A2_test)))["mse"], 4), "with", len(kept), "features")

### Answers (E3)

The proximal step gives **31 exact zeros** (scikit-learn: 32): the lasso selected **13 of the 44** features and its test
MSE is 0.505, better than the plain fit on all 44 (0.533) with a third of the features. Objective and predictions agree
with scikit-learn ($8\cdot10^{-6}$ and $10^{-2}$; the weights differ for the collinearity reason above). The step size
$1/L$ with $L = 23.4$ comes from the curvature of the loss (largest eigenvalue of its Hessian): the same rule as
Class 02, $\eta < 2/\lambda_{\max}$. Soft-thresholding is what makes sparsity possible: a weight whose gradient is
too small to move it beyond the threshold *stays* at 0. Ridge shrinks; lasso shrinks **and selects**.

## Part F — The perceptron

$\hat y = \text{sign}(w^\top x + b)$ with labels $y\in\{-1,+1\}$. Its loss is the **perceptron criterion**
$\ell = \max(0,\,-m)$ with the margin $m = y\,(w^\top x + b)$: zero for a correct point, the distance-like amount
$-m$ for a mistake.

### F1. The loss and the online rule

Implement `perceptron_loss(params, X, y)` (mean over the rows; use `jnp.where(margin <= 0, -margin, 0.0)`, so a point
**on** the boundary counts as a mistake) and `train_perceptron(X, y, epochs)`. It visits the examples **in order**,
computes `jax.grad(perceptron_loss)` of **one example**, and updates $\theta \leftarrow \theta - g$ (learning
rate 1). It returns the parameters and the number of mistakes in every epoch (a mistake is a non-zero gradient).

### How we got there (F1)

For one example with $m \le 0$, $\ell = -y(w^\top x + b)$, so $\nabla_w\ell = -y\,x$ and $\nabla_b\ell = -y$: the
step $\theta - g$ is $w \leftarrow w + yx$, $b \leftarrow b + y$, the classic perceptron rule. For $m > 0$ the gradient is
exactly 0 and nothing moves. We never write $-yx$: `jax.grad` produces it, and a non-zero `g["b"]` tells us there
was a mistake. Using `<=` instead of `<` matters at the very first step, when $w = 0$ and every margin is 0: it makes
this the same rule as scikit-learn's.

In [ ]:
def perceptron_loss(params, X, y) -> jax.Array:
    margin = y * (X @ params["w"] + params["b"])
    return jnp.mean(jnp.where(margin <= 0, -margin, 0.0))


def train_perceptron(X, y, epochs) -> tuple[dict[str, jax.Array], list[int]]:
    grad = jax.jit(jax.grad(perceptron_loss))
    X, y = jnp.asarray(X), jnp.asarray(y, dtype=float)
    params = init_params(X.shape[1])
    mistakes = []
    for _ in range(epochs):
        wrong = 0
        for i in range(len(y)):
            g = grad(params, X[i : i + 1], y[i : i + 1])
            wrong += int(g["b"] != 0)
            params = jax.tree.map(lambda p, gp: p - gp, params, g)
        mistakes.append(wrong)
    return params, mistakes

In [ ]:
def predict_class(params, X):
    return (np.asarray(X @ params["w"] + params["b"]) > 0).astype(int)


fig, axes = plt.subplots(1, 3, figsize=(12, 3.8))
Xb, yb = make_blobs(rng, 100, gap=4.0)
p_sep, mist_sep = train_perceptron(Xb, 2 * yb - 1, 15)
plot_boundary(lambda Z: predict_class(p_sep, Z), Xb, yb, f"separable: mistakes/epoch {mist_sep[:4]}...", ax=axes[0])
Xo, yo = make_blobs(rng, 100, gap=1.5)
p_ov, mist_ov = train_perceptron(Xo, 2 * yo - 1, 15)
plot_boundary(lambda Z: predict_class(p_ov, Z), Xo, yo, f"overlapping: {mist_ov[-3:]} per epoch", ax=axes[1])
Xx, yx = make_xor(rng, 200)
p_xor, mist_xor = train_perceptron(Xx, 2 * yx - 1, 20)
plot_boundary(lambda Z: predict_class(p_xor, Z), Xx, yx, f"XOR: accuracy {np.mean(predict_class(p_xor, Xx) == yx):.2f}", ax=axes[2])
plt.show()
print("separable: mistakes per epoch", mist_sep)
print("overlapping: mistakes per epoch", mist_ov)
print("XOR (raw features): accuracy", np.mean(predict_class(p_xor, Xx) == yx), "| mistakes per epoch", mist_xor[:6], "...")

# XOR becomes separable with the degree-2 features (x1, x2, x1^2, x1 x2, x2^2).
P_xor = polynomial_features(Xx, 2)
p_xp, mist_xp = train_perceptron(P_xor, 2 * yx - 1, 20)
print("XOR with polynomial features: mistakes per epoch", mist_xp, "| accuracy", np.mean(predict_class(p_xp, P_xor) == yx))

In [ ]:
# On spam (standardized log features), one pass over the training set, against scikit-learn's Perceptron.
p_spam, mist_spam = train_perceptron(Xtr_s, 2 * ytr_s - 1, 1)
sk_p = Perceptron(penalty=None, eta0=1.0, shuffle=False, tol=None, max_iter=1).fit(Xtr_s, ytr_s)  # pyright: ignore[reportArgumentType]
compare("perceptron weights", p_spam["w"], np.asarray(sk_p.coef_)[0])
compare("perceptron bias", p_spam["b"], np.asarray(sk_p.intercept_)[0])
print("mistakes in the epoch:", mist_spam[0], "of", len(ytr_s))
print({k: round(v, 3) for k, v in classification_metrics(yte_s, predict_class(p_spam, Xte_s)).items()})

### Answers (F1)

* **The rule.** `jax.grad` of one example returns $-y\,x$ when the margin is $\le 0$ and exactly 0 otherwise, so
  $\theta - g$ is the perceptron rule without our writing it. The result is **identical to scikit-learn's** `Perceptron`
  (difference 0.0 in weights and bias after one epoch on spam).
* **Separable data:** 2 mistakes in the first epoch, then 0 for ever: the perceptron converges, as the convergence
  theorem says when the classes are linearly separable. **Overlapping blobs:** 12 to 18 mistakes in *every* epoch: it never
  converges, and the last boundary is arbitrary. **XOR:** no line separates it; accuracy 0.515 (about 90 to 98 mistakes
  per 200 examples, every epoch: chance level). Adding the degree-2 features (`x1*x2` is the useful one) makes it separable,
  and the perceptron reaches 0 mistakes after 2 epochs (accuracy 1.0). The model is still linear, in the new features.
* **Spam, one epoch:** 370 mistakes out of 3680 training examples; test accuracy 0.907, $F_1$ 0.882, MCC 0.804. A single
  pass with a step of 1 is crude, and logistic regression (next part) does better: it also uses *how wrong* a point is.

## Part G — Logistic regression

$p(y=1\mid x) = \sigma(w^\top x + b)$ with $\sigma(z) = 1/(1+e^{-z})$. Training minimizes the **binary
cross-entropy** (the negative log-likelihood of the labels).
Together with linear regression it is a **generalized linear model** (GLM): a linear predictor, a link (identity,
logit) and a distribution of $y$ (Gaussian, Bernoulli); with the canonical link the gradient is always *features times
residual*.

### G1. Loss, probabilities and regularization

Implement `sigmoid(z)`, `bce_loss(params, X, y)` for labels $y\in\{0,1\}$ and `bce_l2_loss(params, X, y, lam)`
$= \text{BCE} + \frac\lambda2\lVert w\rVert^2$. Write the loss with the *logits* $z$ and `jnp.logaddexp`:

$$\ell = \log(1 + e^{z}) - y\,z$$

Train it on the standardized spam features. scikit-learn's `LogisticRegression(C)` minimizes
$C\sum_i\ell_i + \frac12\lVert w\rVert^2$, so our $\lambda$ corresponds to $C = 1/(\lambda N)$.

### How we got there (G1)

Writing $\log s(z) = z - \log(1+e^z)$ and $\log(1-s(z)) = -\log(1+e^z)$ turns the cross-entropy
$-[y\log s + (1-y)\log(1-s)]$ into $\log(1+e^z) - yz$, which `jnp.logaddexp(0, z)` evaluates without overflow, where
$\log(\sigma(z))$ would underflow to $-\infty$. The L2 version adds $\frac\lambda2\lVert w\rVert^2$ (no bias).
The equivalence with scikit-learn follows by multiplying our objective by $N/\lambda\cdot\lambda$: sklearn's
$C\sum_i \ell_i + \frac12\lVert w\rVert^2 \propto \frac1N\sum_i\ell_i + \frac1{2CN}\lVert w\rVert^2$, so
$\lambda = 1/(CN)$.

In [ ]:
def sigmoid(z) -> jax.Array:
    return 1.0 / (1.0 + jnp.exp(-z))


def bce_loss(params, X, y) -> jax.Array:
    z = X @ params["w"] + params["b"]
    return jnp.mean(jnp.logaddexp(0.0, z) - y * z)


def bce_l2_loss(params, X, y, lam) -> jax.Array:
    return bce_loss(params, X, y) + 0.5 * lam * jnp.sum(params["w"] ** 2)

In [ ]:
lam = 1e-3
log_p, hist_lr = train(bce_l2_loss, init_params(57), Xtr_s, ytr_s.astype(float), lam, lr=0.05, steps=8000)
log_sk = LogisticRegression(C=1 / (lam * len(ytr_s)), tol=1e-10, max_iter=10_000).fit(Xtr_s, ytr_s)
compare("logistic weights", log_p["w"], log_sk.coef_[0])
compare("logistic bias", log_p["b"], log_sk.intercept_[0])
proba = np.asarray(sigmoid(Xte_s @ log_p["w"] + log_p["b"]))
compare("test probabilities", proba, log_sk.predict_proba(Xte_s)[:, 1])
print("ours   ", {k: round(v, 4) for k, v in classification_metrics(yte_s, (proba >= 0.5).astype(int)).items()})
print("sklearn", {k: round(v, 4) for k, v in classification_metrics(yte_s, log_sk.predict(Xte_s)).items()})

### G2. Thresholds and the cost of a mistake

The default threshold 0.5 treats both mistakes alike. Hiding a real e-mail (a false positive) is worse than letting
a spam through. Plot the metrics against the threshold, then pick the **smallest** threshold whose precision on the
*training* set is at least 0.99, and report the confusion matrix and the metrics on the test set.

In [ ]:
plot_thresholds(yte_s, proba, classification_metrics)

proba_tr = np.asarray(sigmoid(Xtr_s @ log_p["w"] + log_p["b"]))
grid = np.linspace(0.05, 0.99, 95)
ok = [t for t in grid if classification_metrics(ytr_s, (proba_tr >= t).astype(int))["precision"] >= 0.99]
t_star = ok[0]
fig, axes = plt.subplots(1, 2, figsize=(8, 3.4))
for ax, t in zip(axes, [0.5, t_star]):
    plot_confusion(*confusion_counts(yte_s, (proba >= t).astype(int)), ax=ax, title=f"threshold {t:.2f}")
plt.show()
for t in (0.5, t_star):
    print(f"threshold {t:.2f}:", {k: round(v, 3) for k, v in classification_metrics(yte_s, (proba >= t).astype(int)).items()})

### Answers (G1–G2)

* **Match.** The logistic weights differ from scikit-learn's by $2\cdot10^{-6}$ and the test probabilities by
  $6\cdot10^{-7}$ (Adam against L-BFGS): the two models make the same predictions. Test set at threshold 0.5:
  **accuracy 0.927, precision 0.918, recall 0.895, $F_1$ 0.907, MCC 0.847**, better than the perceptron (MCC 0.804).
* **Thresholds.** Raising the threshold trades recall for precision. Requiring a *training* precision of 0.99 gives
  the threshold 0.98; on the test set the precision is only **0.974** (below the target: the training set is optimistic
  for the model that was fitted on it) and the recall falls to **0.526** ($F_1$ 0.683, MCC 0.618). Almost half of the spam now
  gets through: that is the price of never hiding a real e-mail. The threshold is a *decision*, chosen with a validation
  set and the costs of the two mistakes; the metric to report is the one that matches the costs.
* **Logistic regression is linear:** on XOR its boundary is a line (accuracy **0.655**), and with the
  degree-2 features it becomes the curved boundary that separates the classes (accuracy **1.0**).

In [ ]:
# Logistic regression is a linear classifier: XOR needs the polynomial features, as for the perceptron.
Px = polynomial_features(Xx, 2)
p_lx, _ = train(bce_l2_loss, init_params(5), Px, yx.astype(float), 1e-3, lr=0.1, steps=2000)
q_lin, _ = train(bce_l2_loss, init_params(2), Xx, yx.astype(float), 1e-3, lr=0.1, steps=2000)
fig, axes = plt.subplots(1, 2, figsize=(8, 3.8))
plot_boundary(lambda Z: predict_class(q_lin, Z), Xx, yx, "logistic, raw features", ax=axes[0])
plot_boundary(lambda Z: predict_class(p_lx, polynomial_features(Z, 2)), Xx, yx, "logistic, degree 2", ax=axes[1])
plt.show()
print("accuracy: raw features", np.mean(predict_class(q_lin, Xx) == yx), "| degree-2 features", np.mean(predict_class(p_lx, Px) == yx))

## Part H — Challenge: One Pipeline, Three Models

The pipeline: stratified 5-fold cross-validation on the training set, every preprocessing step fitted **inside** each
fold, the mean and the standard deviation of the MCC per model, then one final report on the test set. The contenders:
logistic regression on the standardized $\log(1+x)$ features, the same model on the standardized *raw* frequencies, and
the perceptron (2 epochs) on the log features.

In [ ]:
def stratified_folds(y, k, rng):
    per_class = [np.array_split(rng.permutation(np.flatnonzero(y == c)), k) for c in (0, 1)]
    folds = [np.concatenate([pc[i] for pc in per_class]) for i in range(k)]
    return [(np.concatenate([folds[j] for j in range(k) if j != i]), folds[i]) for i in range(k)]


def fit_predict(name, X_fit, y_fit, X_val):
    if name != "logistic, raw frequencies":
        X_fit, X_val = np.log1p(X_fit), np.log1p(X_val)                 # preprocessing inside the fold
    mean, std = fit_standardizer(X_fit)
    Z_fit, Z_val = standardize(X_fit, mean, std), standardize(X_val, mean, std)
    if name == "perceptron":
        p, _ = train_perceptron(Z_fit, 2 * y_fit - 1, 2)
    else:
        p, _ = train(bce_l2_loss, init_params(57), Z_fit, y_fit.astype(float), 1e-3, lr=0.05, steps=1500)
    return predict_class(p, Z_val)


X_tr_raw = X_s[tr_s]
results, fold_scores = {}, {}
for name in ("logistic", "logistic, raw frequencies", "perceptron"):
    scores = []
    for fit_idx, val_idx in stratified_folds(ytr_s, 5, rng):
        y_hat = fit_predict(name, X_tr_raw[fit_idx], ytr_s[fit_idx], X_tr_raw[val_idx])
        scores.append(classification_metrics(ytr_s[val_idx], y_hat)["mcc"])
    results[name], fold_scores[name] = (np.mean(scores), np.std(scores, ddof=1)), scores
    print(f"{name:<26} MCC {results[name][0]:.3f} +- {results[name][1]:.3f}")
plot_cv_scores(fold_scores, "MCC")

### Answers (H)

| model | MCC (5-fold, mean $\pm$ std) |
|:--|:-:|
| logistic regression, log features | **0.877 $\pm$ 0.018** |
| logistic regression, raw frequencies | 0.831 $\pm$ 0.020 |
| perceptron (2 epochs), log features | 0.708 $\pm$ 0.038 |

The logarithm is worth about **4.6 points of MCC**: most frequencies are 0 and a few are huge, so on the raw scale
a handful of e-mails dominate every weight (the same model, the same optimizer). Logistic regression beats the
perceptron by more than four standard deviations of the fold scores; the perceptron also has the largest spread (its result
depends on the order of the examples). The logarithm, the standardizer and every statistic are computed **inside each fold** from
the training part only; nothing from a validation fold reaches a model. The scores of the folds are correlated (the
training sets overlap), so a formal test would use the corrected resampled $t$-test of Class 01.